# Generative Question Answering with T5

This notebook demonstrates how to **fine-tune a T5 model** for a generative (not extractive) question-answering task using Hugging Face Transformers.

### What makes this different from extractive QA?

| Approach | How it works | Example |
|---|---|---|
| **Extractive QA** | Copies a span directly from the context | Context says "Berlin has 3.6 million people" → Answer: "3.6 million" |
| **Generative QA** | Produces a novel answer using the decoder | Context says "Berlin has 3.6 million people" → Answer: "Approximately 3.6 million people live in Berlin" |

We will:
1. Build a small synthetic QA dataset
2. Fine-tune `t5-small` on it
3. Generate answers and inspect the output

---

## 1. Setup

Install the required libraries. If you're running this in Google Colab, these will install into the runtime environment.

In [1]:
!pip install -q transformers datasets accelerate

In [11]:
# !pip freeze

In [2]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from datasets import Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


### A note on the imports

Most imports above are standard, but a few deserve explanation:

- **`AutoModelForSeq2SeqLM`** — A generic loader that picks the right model class based on the checkpoint name. For `t5-small`, it loads a T5 model with a **language modelling head** on top of the decoder (the linear layer that maps hidden states → vocabulary logits).
- **`Seq2SeqTrainer`** — A subclass of `Trainer` that adds support for calling `model.generate()` during evaluation, which is needed because seq2seq models produce output *autoregressively* (token by token), not in a single forward pass.
- **`DataCollatorForSeq2Seq`** — Dynamically pads each batch to the longest sequence *in that batch* (rather than to a global max), which saves memory and speeds up training. It also shifts the decoder labels by one position internally — the standard "teacher forcing" setup where the decoder input at step *t* is the ground-truth token at step *t−1*.
- **`Dataset`** (from `datasets`) — A lightweight wrapper around Arrow tables. We use it because `Trainer` expects this format, and `.map()` makes batch tokenization simple.

## 2. Model Choice — T5-small

**T5 (Text-to-Text Transfer Transformer)** treats every NLP task as a text-to-text problem. It has two main parts:

- **Encoder**: Reads the full input (question + context) and produces a sequence of hidden states — one per input token. These are *bidirectional*: each hidden state "sees" all other tokens via self-attention.
- **Decoder**: Generates the output sequence one token at a time (autoregressively). At each step it attends to two things:
  1. **Causal self-attention** over the tokens it has generated *so far* (masked so it cannot peek at future tokens).
  2. **Cross-attention** over the encoder's hidden states — this is how the decoder "reads" the input while deciding what to generate next.

This cross-attention mechanism is the key difference from decoder-only models (like GPT). It lets the model build a complete understanding of the input first and then focus on generation, which tends to work well for conditional tasks like QA, summarisation, and translation.

We use `t5-small` (60 M parameters) so training stays fast on a free Colab GPU. The architecture is the same as `t5-base` or `t5-large`, just with fewer layers and smaller hidden dimensions.

In [3]:
MODEL_NAME = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

print(f"Model parameters: {model.num_parameters():,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model parameters: 60,506,624


## 3. Synthetic QA Dataset

We create a small dataset directly in the notebook. Each example has:
- A **context** paragraph
- A **question** about the context
- A **target answer** that is intentionally *paraphrased or inferred* — not a verbatim copy from the context

This teaches the model to generate rather than extract.

In [4]:
raw_data = [
    {
        "context": "The Eiffel Tower was constructed in 1889 as the entrance arch for the World's Fair. It stands 330 metres tall and is located in Paris, France.",
        "question": "When was the Eiffel Tower built?",
        "answer": "It was built in 1889, originally serving as the gateway to the World's Fair.",
    },
    {
        "context": "Photosynthesis is the process by which green plants convert sunlight, water, and carbon dioxide into glucose and oxygen.",
        "question": "What is photosynthesis?",
        "answer": "Photosynthesis is how plants use sunlight along with water and CO2 to produce their own food and release oxygen.",
    },
    {
        "context": "Python was created by Guido van Rossum and first released in 1991. It emphasizes code readability and supports multiple programming paradigms.",
        "question": "Who created Python?",
        "answer": "Guido van Rossum is the creator of Python, which he first released in 1991.",
    },
    {
        "context": "The mitochondria are often referred to as the powerhouses of the cell because they generate most of the cell's supply of adenosine triphosphate (ATP).",
        "question": "Why are mitochondria called the powerhouses of the cell?",
        "answer": "They earn this name because they are responsible for producing the majority of the cell's energy in the form of ATP.",
    },
    {
        "context": "The Great Wall of China stretches over 13,000 miles. It was built over many centuries by various Chinese dynasties to protect against northern invasions.",
        "question": "How long is the Great Wall of China?",
        "answer": "The wall extends more than 13,000 miles across China, constructed over centuries as a defensive barrier.",
    },
    {
        "context": "Water boils at 100 degrees Celsius at standard atmospheric pressure. At higher altitudes, the boiling point decreases because atmospheric pressure is lower.",
        "question": "At what temperature does water boil?",
        "answer": "Under normal atmospheric conditions water boils at 100°C, though this temperature drops at higher elevations.",
    },
    {
        "context": "Jupiter is the largest planet in our solar system. It is a gas giant with a mass more than twice that of all the other planets combined.",
        "question": "Which planet is the largest in the solar system?",
        "answer": "Jupiter holds that title — it is a massive gas giant weighing more than double all other planets put together.",
    },
    {
        "context": "Antibiotics are medicines used to treat bacterial infections. They work by killing bacteria or preventing them from reproducing.",
        "question": "How do antibiotics work?",
        "answer": "Antibiotics fight bacterial infections either by destroying the bacteria directly or by stopping them from multiplying.",
    },
    {
        "context": "Leonardo da Vinci painted the Mona Lisa in the early 16th century. The painting is housed in the Louvre Museum in Paris.",
        "question": "Where is the Mona Lisa displayed?",
        "answer": "The Mona Lisa can be seen at the Louvre Museum in Paris, where it has been on display for centuries.",
    },
    {
        "context": "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.",
        "question": "What is machine learning?",
        "answer": "It is a branch of AI where systems automatically learn patterns from data and get better over time without manual coding.",
    },
    {
        "context": "The Amazon River is approximately 6,400 km long, making it the second longest river in the world after the Nile.",
        "question": "How long is the Amazon River?",
        "answer": "The Amazon stretches roughly 6,400 kilometres, ranking as the world's second longest river.",
    },
    {
        "context": "DNS stands for Domain Name System. It translates human-readable domain names like example.com into IP addresses that computers use to communicate.",
        "question": "What does DNS do?",
        "answer": "DNS converts website names that people can read into numeric IP addresses so that computers can locate each other on a network.",
    },
    {
        "context": "Insulin is a hormone produced by the pancreas. It regulates blood sugar levels by signaling cells to absorb glucose from the bloodstream.",
        "question": "What role does insulin play in the body?",
        "answer": "Insulin is a pancreatic hormone that controls blood sugar by telling cells to take in glucose from the blood.",
    },
    {
        "context": "The speed of light in a vacuum is approximately 299,792 kilometres per second. Nothing with mass can travel at or above this speed.",
        "question": "How fast does light travel?",
        "answer": "Light moves at nearly 300,000 km/s in a vacuum, which is the ultimate speed limit in the universe.",
    },
    {
        "context": "Tesla was founded in 2003 by Martin Eberhard and Marc Tarpenning. Elon Musk joined the company in 2004 as chairman of the board.",
        "question": "Who founded Tesla?",
        "answer": "Tesla was originally founded by Martin Eberhard and Marc Tarpenning in 2003, with Elon Musk joining shortly after.",
    },
    {
        "context": "Black holes are regions of spacetime where gravity is so strong that nothing, not even light, can escape from within the event horizon.",
        "question": "What is a black hole?",
        "answer": "A black hole is an area in space with gravitational pull so intense that even light cannot break free once past its boundary.",
    },
]

print(f"Dataset size: {len(raw_data)} examples")
print(f"\nSample entry:")
for k, v in raw_data[0].items():
    print(f"  {k}: {v}")

Dataset size: 16 examples

Sample entry:
  context: The Eiffel Tower was constructed in 1889 as the entrance arch for the World's Fair. It stands 330 metres tall and is located in Paris, France.
  question: When was the Eiffel Tower built?
  answer: It was built in 1889, originally serving as the gateway to the World's Fair.


### Format into T5 instruction prompts

T5 was pre-trained with explicit **task prefixes** like `"translate English to German:"`, `"summarize:"`, etc. The model learned to associate these prefixes with different behaviours. By using `"question: ... context: ..."` we follow this convention — the prefix tells the model *what kind of task* to perform.

This is fundamentally different from BERT-style models where the task type is encoded by the model head (a classification layer, a span-extraction layer, etc.). In T5, the task is encoded *in the input text itself*, and the same decoder head handles everything. This is why it's called "text-to-text".

In [5]:
def format_example(example):
    return {
        "input_text": f"question: {example['question']} context: {example['context']}",
        "target_text": example["answer"],
    }

formatted = [format_example(ex) for ex in raw_data]
dataset = Dataset.from_list(formatted)

print("Input:  ", dataset[0]["input_text"][:100], "...")
print("Target: ", dataset[0]["target_text"])

Input:   question: When was the Eiffel Tower built? context: The Eiffel Tower was constructed in 1889 as the  ...
Target:  It was built in 1889, originally serving as the gateway to the World's Fair.


## 4. Preprocessing / Tokenization

Tokenization converts raw text into integer token IDs that the model can process. There are a few non-obvious details specific to seq2seq models:

**Why `text_target=` instead of just calling `tokenizer()` again?**

T5 uses the same vocabulary for encoder and decoder, so the token IDs are the same either way. However, the `text_target` keyword signals to the tokenizer that these tokens belong to the *decoder side*. In some model families (e.g., mBART) the encoder and decoder use different special tokens or language codes — using `text_target` ensures the right ones are applied. It's a good habit even when it doesn't technically matter.

**Why replace padding with `-100` in labels?**

PyTorch's `CrossEntropyLoss` has a special `ignore_index` parameter (default: `-100`). Any position in the labels set to `-100` is excluded from the loss calculation. Without this, the model would be penalised for not predicting `<pad>` tokens — which would waste capacity learning to output padding instead of meaningful text.

**`max_length` and truncation**

We set `MAX_INPUT_LEN=256` and `MAX_TARGET_LEN=64`. Inputs longer than 256 tokens get truncated (information loss), while shorter ones get padded (wasted compute). In production you would analyse your data distribution to pick these values carefully.

In [6]:
MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 64

def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LEN,
        padding="max_length",
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LEN,
        padding="max_length",
        truncation=True,
    )
    # Replace pad token ids in labels with -100 so they are ignored by the loss
    label_ids = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in seq]
        for seq in labels["input_ids"]
    ]
    model_inputs["labels"] = label_ids
    return model_inputs

tokenized_dataset = dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=dataset.column_names,
)

print("Columns:", tokenized_dataset.column_names)
print("First input_ids length:", len(tokenized_dataset[0]["input_ids"]))

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Columns: ['input_ids', 'attention_mask', 'labels']
First input_ids length: 256


### Inspect the tokenized output

Let's peek at one example to see what the model actually receives — this makes the abstraction concrete.

In [7]:
sample = tokenized_dataset[0]

input_ids = sample["input_ids"]
label_ids = sample["labels"]

# Show the first 30 tokens (before padding takes over)
non_pad_input = [t for t in input_ids if t != tokenizer.pad_token_id]
non_ignored_labels = [t for t in label_ids if t != -100]

print("INPUT (first 30 token IDs):", input_ids[:30])
print("  → decoded:", tokenizer.decode(non_pad_input))
print()
print("LABELS (non-padding IDs):", non_ignored_labels)
print("  → decoded:", tokenizer.decode(non_ignored_labels))
print()
print(f"Input sequence length : {len(input_ids)} (of which {len(non_pad_input)} are real tokens)")
print(f"Label sequence length : {len(label_ids)} (of which {len(non_ignored_labels)} are real tokens)")

INPUT (first 30 token IDs): [822, 10, 366, 47, 8, 6628, 16387, 10677, 1192, 58, 2625, 10, 37, 6628, 16387, 10677, 47, 8520, 16, 507, 3914, 38, 8, 6143, 11508, 21, 8, 1150, 31, 7]
  → decoded: question: When was the Eiffel Tower built? context: The Eiffel Tower was constructed in 1889 as the entrance arch for the World's Fair. It stands 330 metres tall and is located in Paris, France.</s>

LABELS (non-padding IDs): [94, 47, 1192, 16, 507, 3914, 6, 5330, 3122, 38, 8, 17090, 12, 8, 1150, 31, 7, 4506, 5, 1]
  → decoded: It was built in 1889, originally serving as the gateway to the World's Fair.</s>

Input sequence length : 256 (of which 47 are real tokens)
Label sequence length : 64 (of which 20 are real tokens)


## 5. Training

We use `Seq2SeqTrainer` with a set of training arguments. Here's what the non-obvious ones do:

| Argument | Value | Why |
|---|---|---|
| `num_train_epochs=30` | High on purpose — our dataset has only 16 examples, so the model needs many passes to learn the patterns. With a real dataset (thousands+ examples) you'd use 3–5 epochs. |
| `learning_rate=3e-4` | Higher than the typical 5e-5 for BERT fine-tuning. T5 was pre-trained with this LR and responds well to it for fine-tuning too. |
| `weight_decay=0.01` | L2 regularisation applied to all parameters except biases and LayerNorm weights. Helps prevent overfitting by penalising large weight values. |
| `fp16=True` | Mixed-precision training: most operations run in 16-bit floats (faster, less memory), while critical accumulations stay in 32-bit to preserve numerical stability. Only activated when a GPU is available. |
| `predict_with_generate=True` | Tells the trainer to run `model.generate()` during evaluation instead of a simple forward pass. This is essential for seq2seq because a forward pass only gives you loss, not the actual generated text. |
| `save_strategy="no"` | We skip checkpointing to keep the demo fast. In production you'd save periodically. |

**What happens during each training step:**
1. A batch of (input_ids, labels) is fed to the model.
2. The encoder processes input_ids → hidden states.
3. The decoder receives the labels shifted right by one position (teacher forcing) and tries to predict the next token at each position.
4. Cross-entropy loss is computed only on non-padding positions (that's why we set padding to `-100`).
5. Gradients are backpropagated and the AdamW optimiser updates the weights.

In [9]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-qa-output",
    num_train_epochs=30,
    per_device_train_batch_size=8,
    learning_rate=3e-4,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

### Reading the training log

The trainer prints a loss value every `logging_steps` steps. Here's how to interpret it:

- **Loss dropping steadily** → the model is learning the mapping from input to target.
- **Loss plateauing near 0** → the model has essentially memorised the dataset (expected with 16 examples and 30 epochs).
- **Loss oscillating wildly** → learning rate might be too high, or the data is too noisy.

On this tiny dataset, you should see loss drop from ~2–3 down to near 0 within the first few epochs. That's fine for a demo, but on a real task a near-zero training loss with high validation loss would indicate **overfitting**.

## 6. Inference / Generation

During training, the decoder receives the ground-truth tokens as input (teacher forcing). During **inference**, there is no ground truth — the decoder must generate tokens one at a time, feeding each generated token back as input for the next step. This is called **autoregressive decoding**.

At each step the model outputs a probability distribution over the entire vocabulary (~32,000 tokens for T5). How we pick the next token from that distribution is the **decoding strategy**:

### Greedy vs. Sampling

- **Greedy decoding** (`do_sample=False`): Always pick the highest-probability token. Deterministic but often repetitive and boring.
- **Sampling** (`do_sample=True`): Randomly sample from the distribution. More diverse but can be incoherent if unconstrained.

### Temperature — reshaping the distribution

Temperature divides the raw logits (pre-softmax scores) before applying softmax:

$$p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

- **T < 1** → sharpens the distribution (high-prob tokens get even higher prob, low-prob tokens vanish). Output becomes more focused and deterministic.
- **T = 1** → unchanged distribution (model's natural confidence).
- **T > 1** → flattens the distribution (all tokens become more equally likely). Output becomes more random and creative — but also more likely to produce nonsense.

### Top-p (nucleus sampling)

Instead of sampling from the full vocabulary, **top-p** sorts tokens by probability and keeps only the smallest set whose cumulative probability exceeds `p`. For example with `top_p=0.9`:
- If the top 5 tokens already account for 90% of the probability mass, only those 5 are candidates.
- This dynamically adapts: when the model is confident, few tokens are kept; when uncertain, more tokens are included.

This is generally more robust than top-k (fixed number of candidates) because it adapts to the model's confidence at each step.

### Parameters at a glance

| Parameter | What it does |
|---|---|
| `max_new_tokens` | Hard limit on output length (stops generation even if the model hasn't produced `</s>`) |
| `temperature` | Reshapes the probability distribution — lower = focused, higher = diverse |
| `top_p` | Nucleus sampling threshold — dynamically limits the candidate token set |
| `do_sample` | Must be `True` for temperature/top_p to take effect; `False` = greedy |

In [12]:
def generate_answer(question, context, max_new_tokens=64, temperature=0.7, top_p=0.9):
    """Generate an answer given a question and context."""
    input_text = f"question: {question} context: {context}"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

**A note on the code above:**

- `torch.no_grad()` disables gradient computation during inference. This saves memory (no need to store activations for backprop) and speeds things up. Always use it when you're not training.
- `model.generate()` runs the full autoregressive loop internally: encode once, then decode step-by-step until `max_new_tokens` is reached or the model outputs the end-of-sequence token `</s>`.
- `skip_special_tokens=True` in the decoder strips out `<pad>` and `</s>` so you get clean text.

### Quick test on a training example

Let's verify the model learned the pattern by testing on an example it has seen.

In [13]:
ex = raw_data[0]
generated = generate_answer(ex["question"], ex["context"])

print("Question: ", ex["question"])
print("Context:  ", ex["context"])
print("Expected: ", ex["answer"])
print("Generated:", generated)

Question:  When was the Eiffel Tower built?
Context:   The Eiffel Tower was constructed in 1889 as the entrance arch for the World's Fair. It stands 330 metres tall and is located in Paris, France.
Expected:  It was built in 1889, originally serving as the gateway to the World's Fair.
Generated: 1889


## 7. Evaluation Demo

We evaluate in two stages:

**Stage 1 — Training data:** The model should reproduce answers close to the targets. If it can't, training didn't converge. We use `temperature=0.3` (low) here so the output is nearly deterministic, making it easier to compare.

**Stage 2 — Unseen questions:** This is the real test. The model has never seen these (question, context) pairs. With only 16 training examples, `t5-small` won't generalise well — but you should still see coherent attempts, because the pre-trained T5 weights already encode strong language understanding. The fine-tuning mainly taught it the *task format* (given a question and context, produce a fluent answer).

In [14]:
print("=" * 80)
print("RESULTS ON TRAINING DATA")
print("=" * 80)

for i, ex in enumerate(raw_data):
    generated = generate_answer(ex["question"], ex["context"], temperature=0.3)
    print(f"\n--- Example {i+1} ---")
    print(f"Q: {ex['question']}")
    print(f"Expected:  {ex['answer']}")
    print(f"Generated: {generated}")

RESULTS ON TRAINING DATA

--- Example 1 ---
Q: When was the Eiffel Tower built?
Expected:  It was built in 1889, originally serving as the gateway to the World's Fair.
Generated: 1889

--- Example 2 ---
Q: What is photosynthesis?
Expected:  Photosynthesis is how plants use sunlight along with water and CO2 to produce their own food and release oxygen.
Generated: the process by which green plants convert sunlight, water, and carbon dioxide into glucose and oxygen

--- Example 3 ---
Q: Who created Python?
Expected:  Guido van Rossum is the creator of Python, which he first released in 1991.
Generated: Guido van Rossum

--- Example 4 ---
Q: Why are mitochondria called the powerhouses of the cell?
Expected:  They earn this name because they are responsible for producing the majority of the cell's energy in the form of ATP.
Generated: they generate most of the cell's supply of adenosine triphosphate (ATP).

--- Example 5 ---
Q: How long is the Great Wall of China?
Expected:  The wall extend

### Unseen questions — testing generalisation

The examples below were **not** in the training set. We use `temperature=0.5` — slightly higher than above — to give the model a bit more freedom to compose novel sentences. Since the model only learned from 16 examples, it is essentially relying on T5's pre-trained knowledge plus the task format it learned during fine-tuning.

In [15]:
unseen_examples = [
    {
        "question": "What is the capital of France?",
        "context": "France is a country in Western Europe. Its capital city is Paris, which is known for the Eiffel Tower and the Louvre Museum.",
    },
    {
        "question": "Why is exercise important?",
        "context": "Regular physical exercise improves cardiovascular health, strengthens muscles, and boosts mental well-being. Experts recommend at least 150 minutes of moderate activity per week.",
    },
    {
        "question": "What is the boiling point of water at high altitude?",
        "context": "Water boils at 100 degrees Celsius at sea level. At higher altitudes the atmospheric pressure drops, causing water to boil at a lower temperature.",
    },
]

print("=" * 80)
print("RESULTS ON UNSEEN QUESTIONS")
print("=" * 80)

for i, ex in enumerate(unseen_examples):
    generated = generate_answer(ex["question"], ex["context"], temperature=0.5)
    print(f"\n--- Unseen {i+1} ---")
    print(f"Q: {ex['question']}")
    print(f"Context: {ex['context'][:80]}...")
    print(f"Generated: {generated}")

RESULTS ON UNSEEN QUESTIONS

--- Unseen 1 ---
Q: What is the capital of France?
Context: France is a country in Western Europe. Its capital city is Paris, which is known...
Generated: Paris

--- Unseen 2 ---
Q: Why is exercise important?
Context: Regular physical exercise improves cardiovascular health, strengthens muscles, a...
Generated: improves cardiovascular health, strengthens muscles, and boosts mental well-being

--- Unseen 3 ---
Q: What is the boiling point of water at high altitude?
Context: Water boils at 100 degrees Celsius at sea level. At higher altitudes the atmosph...
Generated: 100 degrees Celsius


### Exploring generation parameters — temperature in practice

Below we generate answers to the same question at four different temperatures. Watch for:

- **temp=0.1** → Nearly greedy. The model picks the highest-probability token almost every time. Output is deterministic and "safe".
- **temp=0.5** → Mild diversity. Small variations in word choice, but still coherent.
- **temp=0.9** → Noticeable creativity. The model may rephrase more aggressively or add unexpected details.
- **temp=1.2** → Above 1.0, the distribution is flatter than the model's natural confidence. You may see grammatical errors or hallucinations — the model is "guessing" more freely.

We keep `top_p=0.95` to prevent truly degenerate outputs even at high temperature.

In [16]:
test_q = raw_data[3]["question"]
test_c = raw_data[3]["context"]

print(f"Q: {test_q}\n")

for temp in [0.1, 0.5, 0.9, 1.2]:
    ans = generate_answer(test_q, test_c, temperature=temp, top_p=0.95)
    print(f"  temp={temp:.1f} → {ans}")

Q: Why are mitochondria called the powerhouses of the cell?

  temp=0.1 → they generate most of the cell's supply of adenosine triphosphate (ATP).
  temp=0.5 → they generate most of the cell's supply of adenosine triphosphate (ATP).
  temp=0.9 → they generate most of the cell's supply of adenosine triphosphate (ATP).
  temp=1.2 → they generate most of the cell's supply of adenosine triphosphate (ATP).


## Summary

In this notebook we:

1. **Built** a small synthetic QA dataset with paraphrased answers (not verbatim spans).
2. **Tokenized** the data for T5's encoder-decoder architecture.
3. **Fine-tuned** `t5-small` using `Seq2SeqTrainer`.
4. **Generated** answers with `model.generate()`, controlling creativity via `temperature` and `top_p`.
5. **Compared** expected vs. generated outputs on both training and unseen examples.

### Next steps

- Use a larger dataset (e.g., SQuAD reformulated with paraphrased answers, or MS MARCO).
- Try a bigger model (`t5-base`, `flan-t5-base`) for better generalisation.
- Add ROUGE or BERTScore metrics for quantitative evaluation.
- Experiment with beam search (`num_beams`) vs. sampling strategies.